In [ ]:
# ═══ 0. Colab：从 GitHub 拉取代码并安装环境 ═══
# 首次打开 / 换分支后跑这一格；装完建议 Runtime → Restart session，再从 cell 1 继续。
from __future__ import annotations

import io
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path
from urllib.request import Request, urlopen

# —— 仓库 ——
REPO_OWNER = "Beater-221E"
REPO_NAME = "llm4rec-bias-Integrated"
BRANCH = "main"                 # 分支名；私有仓需 GITHUB_TOKEN
GITHUB_TOKEN = ""               # GitHub PAT（repo 读权限）；公开仓留空
REPO_DIR = Path("/content/llm4rec-bias-Integrated")
FRESH_CLONE = False             # True = 删掉旧目录重新拉
INSTALL_DEPS = True             # pip install requirements + editable
UPGRADE_PIP = True
SKIP_TORCH_PIP = True           # Colab 已自带 torch，跳过 requirements 里的 torch 行

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if not IN_COLAB:
    print("非 Colab，跳过本格（本地用已有仓库 + conda env bias）。")
else:
    if FRESH_CLONE and REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
        print(f"已删除旧目录 {REPO_DIR}")

    need_fetch = not (
        (REPO_DIR / "src" / "llm4rec").exists() and (REPO_DIR / "run.sh").exists()
    )
    if need_fetch:
        zip_url = (
            f"https://codeload.github.com/{REPO_OWNER}/{REPO_NAME}/zip/refs/heads/{BRANCH}"
        )
        headers = {"User-Agent": "llm4rec-colab"}
        if GITHUB_TOKEN:
            headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"
        print(f"下载 {zip_url}")
        with urlopen(Request(zip_url, headers=headers), timeout=180) as resp:
            data = resp.read()
        with zipfile.ZipFile(io.BytesIO(data)) as zf:
            top = zf.namelist()[0].split("/")[0]
            tmp = Path("/content/_llm4rec_unzip")
            if tmp.exists():
                shutil.rmtree(tmp)
            zf.extractall(tmp)
            if REPO_DIR.exists():
                shutil.rmtree(REPO_DIR)
            (tmp / top).rename(REPO_DIR)
            shutil.rmtree(tmp, ignore_errors=True)
        print(f"代码就绪 → {REPO_DIR}")
    else:
        print(f"已有仓库 → {REPO_DIR}（要重拉设 FRESH_CLONE=True）")

    os.chdir(REPO_DIR)
    os.environ["LLM4REC_ROOT"] = str(REPO_DIR)
    src = str(REPO_DIR / "src")
    if src not in sys.path:
        sys.path.insert(0, src)

    if INSTALL_DEPS:
        py = sys.executable
        if UPGRADE_PIP:
            subprocess.check_call(
                [py, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"]
            )
        req = REPO_DIR / "requirements.txt"
        if req.exists():
            lines = req.read_text().splitlines()
            if SKIP_TORCH_PIP:
                skip_pfx = ("torch", "torchaudio", "torchvision")
                kept = []
                for ln in lines:
                    s = ln.strip()
                    if not s or s.startswith("#"):
                        kept.append(ln)
                        continue
                    name = s.split("==")[0].split(">=")[0].split("<=")[0].split("[")[0].strip().lower()
                    if name in skip_pfx:
                        print(f"跳过 requirements 行（Colab 已有）: {s}")
                        continue
                    kept.append(ln)
                with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as f:
                    f.write("\n".join(kept) + "\n")
                    req_path = f.name
            else:
                req_path = str(req)
            subprocess.check_call([py, "-m", "pip", "install", "-q", "-r", req_path])
            if SKIP_TORCH_PIP and req_path != str(req):
                Path(req_path).unlink(missing_ok=True)
        subprocess.check_call([py, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])
        print("依赖安装完成")
        try:
            import torch
            import transformers
            import llm4rec
            print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}")
            print(f"transformers {transformers.__version__}")
            print(f"llm4rec → {list(llm4rec.__path__)[0]}")
        except Exception as e:
            print(f"导入检查失败（可 Restart 后再试）: {e}")

    print("LLM4REC_ROOT =", os.environ["LLM4REC_ROOT"])
    print("★ 若刚装完包：Runtime → Restart session，然后从 cell 1 往下跑")


In [1]:
# ═══ 1. bootstrap（定位 ROOT / sys.path）═══
from __future__ import annotations

import json
import os
import shlex
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path
from typing import Any


def _find_root() -> Path:
    cands: list[Path] = []
    if os.environ.get("LLM4REC_ROOT"):
        cands.append(Path(os.environ["LLM4REC_ROOT"]))
    cands.append(Path("/content/llm4rec-bias-Integrated"))
    here = Path.cwd()
    cands += [here, here.parent]
    for p in [here, *here.parents]:
        cands.append(p)
        if len(cands) > 20:
            break
    seen: set[Path] = set()
    for c in cands:
        try:
            c = c.resolve()
        except Exception:
            continue
        if c in seen:
            continue
        seen.add(c)
        if c.exists() and (c / "src" / "llm4rec").exists() and (c / "run.sh").exists():
            return c
    raise FileNotFoundError(
        "找不到项目根。Colab 请先跑 cell 0（从 GitHub 拉代码并安装）。"
    )


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return "COLAB_RELEASE_TAG" in os.environ


ROOT = _find_root()
os.chdir(ROOT)
SRC = str(ROOT / "src")
sys.path = [p for p in sys.path if "llm4rec" not in p or p == SRC]
sys.path.insert(0, SRC)
os.environ["LLM4REC_ROOT"] = str(ROOT)
os.environ["PYTHONPATH"] = SRC + (":" + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else "")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTHONUNBUFFERED", "1")
IN_COLAB = _in_colab()
print(f"ROOT={ROOT}  colab={IN_COLAB}")


FileNotFoundError: 找不到项目根（需要 src/llm4rec + run.sh）

In [ ]:
# ═══ 2. NB_CFG：notebook 总配置（改这里即可覆写 yaml）═══
# · None / 省略的键 = 不覆写，沿用 compose(EXP) 的 yaml
# · drive.* 只影响本 notebook 的持久化，不会传给 train CLI
# · run.*  只影响本 notebook 的启动方式

NB_CFG: dict[str, Any] = {
    # ── notebook 运行控制（不进 yaml override）──
    "run": {
        "exp": "minionerec_qwen05b_amazon",
        "gpus": "auto",                 # auto | 0 | 0,1,2,3
        "conda_env": "bias",
        "deepspeed": None,              # None | zero2 | zero2_offload | zero3
        "force_prepare": False,
        "run_prepare": False,           # True 才执行 prepare.sh
        "launch_mode": "print",         # print | foreground | background
        # full | sft_only | continue_after_sft | rl_only | eval_only | custom
        "recipe": "continue_after_sft",
        "stages": None,                 # 例 "eval,rl,eval"；None=按 recipe/yaml
        "resume_from": None,            # 路径；None=配方自动找
        "sft_final_hint": None,         # 钉死某次 sft/final
        "run_dir_override": None,
    },

    # ── Google Drive 持久化（不进 yaml override）──
    "drive": {
        # auto=仅 Colab 启用；True=强制（需 root 可写）；False=关闭
        "enabled": "auto",
        "mount": True,                  # Colab 下是否 drive.mount
        "pull": True,                   # ★ 有 Drive 资源则优先使用（软链到本地）
        "push": True,                   # ★ 本地产出写入 Drive（软链，后续写落盘到 Drive）
        "migrate_local_to_drive": True, # Drive 空但本地有 → 先拷到 Drive 再软链
        "root": "/content/drive/MyDrive/llm4rec-bias",
        # 各类资源是否参与 pull/push
        "resources": {
            "raw": True,                # data/raw
            "processed": True,          # data/processed
            "embeddings": True,         # artifacts/embeddings
            "sid": True,                # artifacts/sid
            "bm25": True,               # artifacts/bm25
            "runs": True,               # runs/** 含 sft/rl/final + checkpoint-*
        },
    },

    # ── 以下键与 configs/*.yaml 对齐；设为具体值才会生成 dotted override ──
    "seed": None,

    "experiment": {
        # "name": "minionerec_qwen05b_amazon",
        # "route": "minionerec",
        # "mode": "integrated",
    },

    "stages": None,  # 例: ["sft", "eval", "rl", "eval"]；通常用 run.stages 字符串

    "data": {
        # "category": "Industrial_and_Scientific",
        # "min_uc": 5,
        # "min_sc": 5,
        # "rating_threshold": 4.0,
        # "history_max_length": 20,
        # "max_train_samples": -1,
        # "max_eval_samples": -1,
    },

    "hardware": {
        # "devices": "auto",
        # "precision": "auto",          # V100: 勿用 bf16
        # "strategy": "auto",
        # "memory": "auto",
        # "gradient_checkpointing": True,
        # "deepspeed": None,
    },

    "train": {
        "sft": {
            # "epochs": 3,
            # "max_steps": None,
            # "learning_rate": 1.0e-5,
            # "lr_scheduler_type": "cosine",
            # "warmup_ratio": 0.03,
            # "global_batch_size": 64,
            # "per_device_batch_size": 2,
            # "gradient_accumulation_steps": 8,
            # "max_seq_length": 512,
            # "logging_steps": 10,
            # "eval_steps": 200,
            # "load_best_model_at_end": False,
        },
        "rl": {
            # "epochs": 2,
            # "max_steps": None,
            # "learning_rate": 1.0e-5,
            # "eval_steps": 25,
            # "bias_eval_steps": 25,
            # "eval_examples": 256,
            # "per_device_batch_size": 1,
            # "gradient_accumulation_steps": 8,
            # "global_batch_size": 8,
            # "grpo": {
            #     "group_size": 16,
            #     "beta": 1.0e-3,
            #     "temperature": 1.0,
            # },
        },
    },

    "bias": {
        # "top_k": 10,
        # "final_examples": None,       # 冒烟可改 512
        # "ips_gamma": 1.0,
        # "online_stages": ["rl"],
    },

    "checkpoint": {
        # "save_stage_final": True,
        # "save_steps": 500,            # null 关闭中间存盘
        # "save_total_limit": 3,
        # "save_optimizer_state": False,
    },

    "sid": {
        # "codebook_size": 512,
        # "levels": 3,
        # "implementation": "integrated",
        # "rqvae": {"epochs": 2000, "batch_size": 2048},
    },

    "decoder": {
        # "name": "constrained_beam",
        # "num_beams": 20,
        # "fail_on_invalid": True,
    },

    "wandb": {
        # "enabled": True,
        # "project": "llm4rec-bias",
        # "mode": "online",             # online | offline | disabled
        # "group": "minionerec_qwen05b_amazon",
    },

    "evaluation": {
        # "top_k": [1, 5, 10, 20],
        # "metrics": ["hr", "ndcg", "mrr"],
        # "max_examples": None,
    },

    "optimization": {
        # "compile": {"enabled": "auto"},
        # "attention": {"implementation": "auto"},
    },
}

print("NB_CFG keys:", list(NB_CFG))
print("drive.enabled =", NB_CFG["drive"]["enabled"],
      "pull =", NB_CFG["drive"]["pull"],
      "push =", NB_CFG["drive"]["push"])
print("run.recipe =", NB_CFG["run"]["recipe"], "exp =", NB_CFG["run"]["exp"])

In [ ]:
# ═══ 3. 解析 NB_CFG → OVERRIDES / env；Drive pull·push ═══

NOTEBOOK_ONLY = {"run", "drive"}


def _is_set(v: Any) -> bool:
    return v is not None and v != {} and v != []


def _flatten(prefix: str, value: Any, out: list[str]) -> None:
    if isinstance(value, dict):
        if not value:
            return
        for k, v in value.items():
            key = f"{prefix}.{k}" if prefix else str(k)
            _flatten(key, v, out)
        return
    # 显式 null：在 NB_CFG 里写 "null" / "__null__"
    if value in ("null", "__null__"):
        out.append(f"{prefix}=null")
        return
    if value is None:
        return
    if isinstance(value, bool):
        out.append(f"{prefix}={'true' if value else 'false'}")
    elif isinstance(value, (list, tuple)):
        inner = ",".join(str(x) for x in value)
        out.append(f"{prefix}=[{inner}]")
    else:
        out.append(f"{prefix}={value}")


def nb_cfg_to_overrides(cfg: dict[str, Any]) -> list[str]:
    out: list[str] = []
    for k, v in cfg.items():
        if k in NOTEBOOK_ONLY:
            continue
        if not _is_set(v) and v != 0 and v is not False:
            # allow explicit False / 0
            if v is False or v == 0:
                _flatten(k, v, out)
            continue
        _flatten(k, v, out)
    return out


def _dir_has_payload(path: Path, markers: tuple[str, ...]) -> bool:
    if not path.exists():
        return False
    if path.is_symlink() and not path.exists():
        return False
    for m in markers:
        if list(path.rglob(m)):
            return True
    # fallback: any file
    return any(p.is_file() for p in path.rglob("*"))


RESOURCE_SPECS = {
    "raw": {"local": "data/raw", "markers": ("*.jsonl.gz", "*.csv", "*.tsv", "*.gz")},
    "processed": {"local": "data/processed", "markers": ("interactions.jsonl", "item_meta.json", "stats.json")},
    "embeddings": {"local": "artifacts/embeddings", "markers": ("item_emb.npy",)},
    "sid": {"local": "artifacts/sid", "markers": ("item2sid.json", "manifest.json")},
    "bm25": {"local": "artifacts/bm25", "markers": ("*.pkl", "*.json", "index*")},
    "runs": {
        "local": "runs",
        "markers": ("model.safetensors", "model*.safetensors", "pytorch_model.bin", "summary.json", "resolved_config.json"),
    },
}


def _resolve_drive_enabled(drive_cfg: dict[str, Any]) -> bool:
    flag = drive_cfg.get("enabled", "auto")
    if flag is True or flag == "true" or flag == 1:
        return True
    if flag is False or flag in {"false", 0, None}:
        return False
    # auto
    return bool(IN_COLAB)


def _ensure_drive_mounted(drive_cfg: dict[str, Any]) -> Path | None:
    root = Path(drive_cfg.get("root") or "/content/drive/MyDrive/llm4rec-bias")
    if not _resolve_drive_enabled(drive_cfg):
        print("[drive] disabled")
        return None
    if drive_cfg.get("mount", True) and IN_COLAB:
        from google.colab import drive as gdrive

        gdrive.mount("/content/drive", force_remount=False)
    if not root.parent.exists() and not root.exists():
        print(f"[drive] root parent missing: {root.parent} — skip")
        return None
    root.mkdir(parents=True, exist_ok=True)
    print(f"[drive] root = {root}")
    return root


def _replace_with_symlink(local: Path, target: Path) -> None:
    local.parent.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        if local.resolve() == target.resolve():
            return
        local.unlink()
    elif local.exists():
        bak = local.with_name(local.name + ".local_bak")
        if bak.exists():
            shutil.rmtree(bak) if bak.is_dir() and not bak.is_symlink() else bak.unlink()
        local.rename(bak)
        print(f"  local backup → {bak.name}")
    local.symlink_to(target, target_is_directory=True)


def sync_drive_resources(root: Path, drive_cfg: dict[str, Any]) -> None:
    pull = bool(drive_cfg.get("pull", True))
    push = bool(drive_cfg.get("push", True))
    migrate = bool(drive_cfg.get("migrate_local_to_drive", True))
    enabled_res = drive_cfg.get("resources") or {}
    print(f"[drive] pull={pull} push={push} migrate={migrate}")

    for name, spec in RESOURCE_SPECS.items():
        if not enabled_res.get(name, True):
            print(f"  · {name}: skipped (resources.{name}=false)")
            continue
        local = ROOT / spec["local"]
        remote = root / spec["local"]
        remote.mkdir(parents=True, exist_ok=True)
        markers = tuple(spec["markers"])
        local_ok = _dir_has_payload(local, markers) if local.exists() else False
        remote_ok = _dir_has_payload(remote, markers)

        if pull and remote_ok:
            _replace_with_symlink(local, remote)
            print(f"  ✓ {name}: USE drive ({remote})")
            continue

        if push and local_ok and not remote_ok and migrate and not local.is_symlink():
            print(f"  → {name}: migrate local → drive …")
            # copytree into remote
            for item in local.iterdir():
                dst = remote / item.name
                if item.is_dir():
                    shutil.copytree(item, dst, dirs_exist_ok=True)
                else:
                    shutil.copy2(item, dst)
            _replace_with_symlink(local, remote)
            print(f"  ✓ {name}: migrated + linked → {remote}")
            continue

        if push:
            # ensure future writes go to drive
            if local.is_symlink() and local.resolve() == remote.resolve():
                print(f"  · {name}: already linked → drive")
            elif not local_ok and not remote_ok:
                _replace_with_symlink(local, remote)
                print(f"  · {name}: empty link → drive (will persist later writes)")
            elif local_ok and remote_ok and not local.is_symlink():
                print(f"  ⚠ {name}: local & drive both nonempty; pull=false or conflict — keep local")
            else:
                print(f"  · {name}: local_ok={local_ok} remote_ok={remote_ok}")
        else:
            print(f"  · {name}: local_ok={local_ok} remote_ok={remote_ok} (push off)")


RUN = NB_CFG["run"]
DRIVE = NB_CFG["drive"]
EXP = str(RUN["exp"])
GPUS = str(RUN.get("gpus") or "auto")
CONDA_ENV = str(RUN.get("conda_env") or "bias")
DEEPSPEED = RUN.get("deepspeed")
FORCE_PREPARE = bool(RUN.get("force_prepare", False))
RUN_PREPARE = bool(RUN.get("run_prepare", False))
LAUNCH_MODE = str(RUN.get("launch_mode") or "print")
RECIPE = str(RUN.get("recipe") or "custom")
SFT_FINAL_HINT = RUN.get("sft_final_hint") or ""
RUN_DIR_OVERRIDE = RUN.get("run_dir_override") or ""

OVERRIDES = nb_cfg_to_overrides(NB_CFG)
if DEEPSPEED:
    OVERRIDES.append(f"hardware.deepspeed={DEEPSPEED}")

wandb_mode = None
for o in OVERRIDES:
    if o.startswith("wandb.mode="):
        wandb_mode = o.split("=", 1)[1]
os.environ["WANDB_MODE"] = wandb_mode or os.environ.get("WANDB_MODE", "online")
os.environ["WANDB_PROJECT"] = os.environ.get("WANDB_PROJECT", "llm4rec-bias")
if GPUS != "auto":
    os.environ["CUDA_VISIBLE_DEVICES"] = GPUS
else:
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)

DRIVE_ROOT = _ensure_drive_mounted(DRIVE)
if DRIVE_ROOT is not None:
    sync_drive_resources(DRIVE_ROOT, DRIVE)

print("\nOVERRIDES:")
if OVERRIDES:
    for o in OVERRIDES:
        print(" ", o)
else:
    print("  <none — 使用 yaml 默认>")
print(f"EXP={EXP} GPUS={GPUS} RECIPE={RECIPE} WANDB_MODE={os.environ['WANDB_MODE']}")

In [ ]:
# ═══ 4. 环境自检 ═══
print(f"Python {sys.version.split()[0]}")
try:
    print(subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.used,utilization.gpu",
         "--format=csv,noheader"], text=True
    ).strip())
except Exception as e:
    print(f"nvidia-smi: {e}")
try:
    import torch
    print(f"torch {torch.__version__} cuda={torch.cuda.is_available()} n={torch.cuda.device_count()}")
except Exception as e:
    print(f"torch: {e}")
import llm4rec
print("llm4rec:", list(llm4rec.__path__)[0])

In [ ]:
# ═══ 5. 状态面板（本地 + Drive 资源）═══
from llm4rec.core.compose import compose, to_dict, validate
from llm4rec.data.base import get_adapter


def _human_bytes(n: int) -> str:
    x = float(n)
    for u in ("B", "KB", "MB", "GB", "TB"):
        if x < 1024:
            return f"{x:.0f}{u}" if u == "B" else f"{x:.1f}{u}"
        x /= 1024
    return f"{x:.1f}PB"


def _has_weights(d: Path) -> bool:
    return d.is_dir() and (
        any(d.glob("model*.safetensors")) or (d / "pytorch_model.bin").exists()
    )


def find_latest_sft_final(root: Path) -> Path | None:
    cands = [p for p in (root / "runs").rglob("sft/final") if _has_weights(p)]
    if not cands:
        return None
    cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return cands[0]


def find_run_dirs(root: Path, limit: int = 8) -> list[Path]:
    runs = root / "runs"
    if not runs.exists():
        return []
    dirs = [
        p for p in runs.rglob("*")
        if p.is_dir()
        and len(p.name) >= 15
        and p.name[0].isdigit()
        and "_" in p.name
        and (
            (p / "sft").exists()
            or (p / "rl").exists()
            or (p / "eval").exists()
            or (p / "resolved_config.json").exists()
        )
    ]
    dirs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return dirs[:limit]


def summarize_run(run: Path) -> dict[str, Any]:
    info: dict[str, Any] = {
        "run": str(run.relative_to(ROOT)),
        "sft": "-",
        "rl": "-",
        "evals": 0,
        "mid": 0,
    }
    if _has_weights(run / "sft" / "final"):
        info["sft"] = "final✓"
    elif (run / "sft").exists():
        info["sft"] = "partial"
    if _has_weights(run / "rl" / "final"):
        info["rl"] = "final✓"
    elif (run / "rl").exists():
        info["rl"] = "partial"
    if (run / "eval").exists():
        info["evals"] = len(list((run / "eval").glob("eval_*.json")))
    info["mid"] = len(list((run / "sft").glob("checkpoint-*"))) + len(
        list((run / "rl").glob("checkpoint-*"))
    )
    return info


cfg_probe = validate(to_dict(compose(EXP, list(OVERRIDES))))
ROUTE = cfg_probe["experiment"]["route"]
NEEDS_SID = "sid" in cfg_probe
NEEDS_BM25 = str((cfg_probe.get("decoder") or {}).get("name")) == "bm25_query"
adapter = get_adapter(cfg_probe)
proc_dir = Path(adapter.processed_dir(cfg_probe))
ds_key = adapter.dataset_key(cfg_probe)

print("═══ local resources ═══")
for name, spec in RESOURCE_SPECS.items():
    local = ROOT / spec["local"]
    ok = _dir_has_payload(local, tuple(spec["markers"])) if local.exists() else False
    link = f" → {local.resolve()}" if local.is_symlink() else ""
    print(f"  {'✓' if ok else '✗'} {name:12s} {local}{link}")

if DRIVE_ROOT is not None:
    print("\n═══ drive resources ═══")
    for name, spec in RESOURCE_SPECS.items():
        remote = DRIVE_ROOT / spec["local"]
        ok = _dir_has_payload(remote, tuple(spec["markers"])) if remote.exists() else False
        print(f"  {'✓' if ok else '✗'} {name:12s} {remote}")

print("\n═══ recent runs ═══")
runs = find_run_dirs(ROOT)
if not runs:
    print("(尚无 runs)")
else:
    print(f"{'run':55s}  sft     rl      evals  mid")
    for r in runs:
        s = summarize_run(r)
        print(f"{s['run']:55s}  {s['sft']:6s}  {s['rl']:6s}  {s['evals']:5d}  {s['mid']:3d}")

LATEST_SFT = Path(SFT_FINAL_HINT) if SFT_FINAL_HINT else find_latest_sft_final(ROOT)
if LATEST_SFT and not LATEST_SFT.is_absolute():
    LATEST_SFT = (ROOT / LATEST_SFT).resolve()
print("\n═══ resume candidate ═══")
if LATEST_SFT and LATEST_SFT.exists():
    w = next(LATEST_SFT.glob("model*.safetensors"), None) or (LATEST_SFT / "pytorch_model.bin")
    sz = w.stat().st_size if w.exists() else 0
    print(f"SFT final → {LATEST_SFT}  ({_human_bytes(sz)})")
else:
    print("未找到 sft/final")
    LATEST_SFT = None
print(f"route={ROUTE} dataset={ds_key} sid={NEEDS_SID} bm25={NEEDS_BM25}")

In [ ]:
# ═══ 6. 校验配置 ═══
from llm4rec.cli.main import _print_plan, cmd_list

cmd_list()
print()
cfg = validate(to_dict(compose(EXP, list(OVERRIDES))))
_print_plan(cfg)
ckpt = cfg.get("checkpoint") or {}
print(
    f"checkpoint.save_steps={ckpt.get('save_steps')} "
    f"limit={ckpt.get('save_total_limit')} stages={cfg.get('stages')}"
)

In [ ]:
# ═══ 7. prepare.sh ═══
sid_ok = (not NEEDS_SID) or _dir_has_payload(ROOT / "artifacts" / "sid", ("item2sid.json",))
ready = proc_dir.exists() and sid_ok
if not RUN_PREPARE:
    print("跳过 prepare（run.run_prepare=False）" + ("；产物已就绪" if ready else "；⚠ 产物未齐"))
elif ready and not FORCE_PREPARE:
    print("产物已就绪，跳过；强制重建请设 run.force_prepare=True")
else:
    env = os.environ.copy()
    env["EXP"] = EXP
    env["GPUS"] = "0" if GPUS == "auto" else GPUS.split(",")[0].strip()
    env["FORCE"] = "1" if FORCE_PREPARE else "0"
    env["CONDA_ENV"] = CONDA_ENV
    cmd = ["bash", str(ROOT / "prepare.sh"), *OVERRIDES]
    print("+", " ".join(shlex.quote(c) for c in cmd))
    rc = subprocess.run(cmd, cwd=str(ROOT), env=env).returncode
    if rc:
        raise RuntimeError(f"prepare 失败 exit={rc}")
    print("prepare 完成 ✓")

In [ ]:
# ═══ 8. 解析 recipe → STAGES / RESUME_FROM ═══
recipe = RECIPE.strip().lower()
stages = (RUN.get("stages") or "")
stages = stages.strip() if isinstance(stages, str) else ",".join(stages)
resume = (RUN.get("resume_from") or "")
resume = resume.strip() if isinstance(resume, str) else str(resume)

if recipe == "full":
    stages, resume = "", ""
elif recipe == "sft_only":
    stages, resume = "sft", ""
elif recipe == "continue_after_sft":
    stages = stages or "eval,rl,eval"
    resume = resume or (str(LATEST_SFT) if LATEST_SFT else "")
    if not resume:
        raise FileNotFoundError("continue_after_sft 需要 sft/final")
elif recipe == "rl_only":
    stages = stages or "rl,eval"
    resume = resume or (str(LATEST_SFT) if LATEST_SFT else "")
    if not resume:
        raise FileNotFoundError("rl_only 需要 sft/final")
elif recipe == "eval_only":
    stages = stages or "eval"
    resume = resume or (str(LATEST_SFT) if LATEST_SFT else "")
    if not resume:
        raise FileNotFoundError("eval_only 需要 resume_from / sft/final")
elif recipe != "custom":
    raise ValueError(f"未知 recipe={RECIPE!r}")

STAGES_EFF, RESUME_EFF = stages, resume
print(f"RECIPE={recipe}")
print(f"STAGES={STAGES_EFF or '(yaml)'}")
print(f"RESUME_FROM={RESUME_EFF or '(none)'}")
if RESUME_EFF:
    rp = Path(RESUME_EFF)
    if not rp.is_absolute():
        rp = ROOT / rp
    assert rp.exists(), rp
    print("resume OK:", rp)

In [ ]:
# ═══ 9. 组装 run.sh ═══
env_run = os.environ.copy()
env_run["EXP"] = EXP
env_run["GPUS"] = GPUS
env_run["WANDB_MODE"] = os.environ.get("WANDB_MODE", "online")
env_run["CONDA_ENV"] = CONDA_ENV
env_run["STAGES"] = STAGES_EFF
env_run["RESUME_FROM"] = RESUME_EFF
env_run["DEEPSPEED"] = str(DEEPSPEED or "")

run_cmd = ["bash", str(ROOT / "run.sh"), *OVERRIDES]
preview = (
    f"EXP={shlex.quote(EXP)} GPUS={shlex.quote(GPUS)} "
    f"WANDB_MODE={shlex.quote(env_run['WANDB_MODE'])} "
    + (f"STAGES={shlex.quote(STAGES_EFF)} " if STAGES_EFF else "")
    + (f"RESUME_FROM={shlex.quote(RESUME_EFF)} " if RESUME_EFF else "")
    + (f"DEEPSPEED={shlex.quote(str(DEEPSPEED))} " if DEEPSPEED else "")
    + "bash run.sh"
    + (" " + " ".join(shlex.quote(o) for o in OVERRIDES) if OVERRIDES else "")
)
print(preview)

In [ ]:
# ═══ 10. 启动（由 run.launch_mode 控制）═══
logs = ROOT / "logs"
logs.mkdir(exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = logs / f"{EXP}_{ts}.log"
pid_path = logs / f"{EXP}_{ts}.pid"

if LAUNCH_MODE == "print":
    print("未启动（launch_mode=print）。复制上一格命令到终端/tmux。")
elif LAUNCH_MODE == "foreground":
    print("log →", log_path)
    with open(log_path, "w") as f:
        p = subprocess.Popen(
            run_cmd, cwd=str(ROOT), env=env_run,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        assert p.stdout
        for line in p.stdout:
            print(line, end="")
            f.write(line)
            f.flush()
        rc = p.wait()
    if rc:
        raise RuntimeError(f"run.sh exit={rc} log={log_path}")
    print("完成 ✓")
elif LAUNCH_MODE == "background":
    with open(log_path, "w") as f:
        p = subprocess.Popen(
            run_cmd, cwd=str(ROOT), env=env_run,
            stdout=f, stderr=subprocess.STDOUT, start_new_session=True,
        )
    pid_path.write_text(str(p.pid) + "\n")
    print(f"background pid={p.pid}\nlog → {log_path}\npid → {pid_path}")
else:
    raise ValueError(LAUNCH_MODE)
LAST_LOG = log_path

In [ ]:
# ═══ 11. 盯日志 / GPU ═══
TAIL_N = 40
cands = sorted((ROOT / "logs").glob(f"{EXP}_*.log"), key=lambda p: p.stat().st_mtime, reverse=True)
log_file = cands[0] if cands else None
print("latest log:", log_file)
try:
    print(subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,utilization.gpu,memory.used,memory.total",
         "--format=csv,noheader"], text=True
    ).strip())
except Exception:
    pass
if log_file and log_file.exists():
    lines = log_file.read_text(errors="replace").splitlines()
    print(f"\n── tail {TAIL_N}/{len(lines)} ──")
    for line in lines[-TAIL_N:]:
        print(line)
    keys = ("stage:", "mid-checkpoint", "完成，权重", "bias_delta", "Error", "Traceback", "constrained_beam", "评测")
    hit = [ln for ln in lines if any(k in ln for k in keys)]
    if hit:
        print("\n── key lines ──")
        for ln in hit[-15:]:
            print(ln)

In [ ]:
# ═══ 12. RUN_DIR + checkpoints ═══
if RUN_DIR_OVERRIDE:
    RUN_DIR = Path(RUN_DIR_OVERRIDE)
    if not RUN_DIR.is_absolute():
        RUN_DIR = ROOT / RUN_DIR
else:
    recent = find_run_dirs(ROOT, limit=1)
    if not recent:
        raise FileNotFoundError("没有 runs")
    RUN_DIR = recent[0]

print("RUN_DIR =", RUN_DIR)
for sub in ("sft", "rl", "dpo", "eval"):
    p = RUN_DIR / sub
    if not p.exists():
        continue
    print(f"\n[{sub}]")
    for child in sorted(p.iterdir()):
        if child.is_dir():
            print(f"  {'✓' if _has_weights(child) else '·'} {child.name}")
        elif child.suffix == ".json":
            print(f"  · {child.name}")
summary_path = RUN_DIR / "summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    for k, v in summary.items():
        if isinstance(v, dict) and v.get("checkpoint"):
            print(f"summary {k}: {v['checkpoint']}")
else:
    print("(尚无 summary.json)")

In [ ]:
# ═══ 13. bias / eval 指标 ═══
KEYS = [
    "hr@10", "ndcg@10", "hr_ips@10", "ndcg_ips@10",
    "pop_lift@1", "pop_lift@10", "delta_gap",
    "exposure_gini", "coverage@10", "tier_gap",
    "history_copy_rate", "top1_concentration", "valid_rate",
]
eval_dir = RUN_DIR / "eval"
files = sorted(eval_dir.glob("eval_*.json")) if eval_dir.exists() else []
if not files:
    print(f"尚无 {eval_dir}/eval_*.json")
else:
    for ef in files:
        payload = json.loads(ef.read_text())
        m = payload.get("metrics") or {}
        print(f"\n── {ef.name}  ckpt={payload.get('checkpoint', '')} ──")
        for k in KEYS:
            if k in m:
                v = m[k]
                print(f"  {k:22s} {v:.6f}" if isinstance(v, float) else f"  {k:22s} {v}")
    delta_path = eval_dir / "bias_delta.json"
    if delta_path.exists():
        delta = json.loads(delta_path.read_text())
        print("\n── bias_delta ──")
        for k in KEYS:
            dk = f"delta/{k}"
            if dk in delta:
                print(f"  {dk:28s} {delta[dk]:+.6f}")